In [1]:
import pandas as pd
import numpy as np

def compare_imputation(df_raw, df_clean):
    """
    So sánh giá trị Null trước/sau và chỉ tính thống kê trên các cột Metrics.
    """
    # Danh sách Metrics cần tính thống kê
    metric_cols = [
        'shortwave_radiation', 'direct_normal_irradiance', 'diffuse_solar_radiation',
        'temperature_c', 'cloud_cover_total', 'cloud_cover_low', 'cloud_cover_mid', 
        'cloud_cover_high', 'wind_speed', 'precipitation_mm', 'sunshine_duration'
    ]
    
    # Chỉ lấy những cột thực sự tồn tại trong DataFrame
    common_metrics = [c for c in metric_cols if c in df_raw.columns and c in df_clean.columns]
    
    # 1. Tổng kết Null (áp dụng cho tất cả cột để kiểm tra coverage)
    all_cols = [c for c in df_raw.columns if c in df_clean.columns]
    summary = pd.DataFrame({
        'Null_Before': df_raw[all_cols].isnull().sum(),
        'Null_After': df_clean[all_cols].isnull().sum()
    })
    summary['Filled_Count'] = summary['Null_Before'] - summary['Null_After']
    
    print("--- TỔNG KẾT SỐ LƯỢNG GIÁ TRỊ ĐÃ FILL ---")
    print(summary[summary['Filled_Count'] > 0].to_string())
    
    # 2. So sánh Mean chỉ trên các cột Metrics
    print("\n--- SO SÁNH MEAN (CHỈ CÁC CỘT METRIC) ---")
    stats_compare = pd.DataFrame({
        'Mean_Before': df_raw[common_metrics].mean(numeric_only=True),
        'Mean_After': df_clean[common_metrics].mean(numeric_only=True)
    })
    # Thêm cột % chênh lệch để dễ đánh giá độ lệch phân phối
    stats_compare['Diff_%'] = ((stats_compare['Mean_After'] - stats_compare['Mean_Before']) / stats_compare['Mean_Before'] * 100).abs()
    print(stats_compare.to_string())
    
    # 3. Mẫu thay đổi
    print("\n--- MẪU DỮ LIỆU ĐÃ THAY ĐỔI ---")
    for col in common_metrics:
        if summary.loc[col, 'Filled_Count'] > 0:
            mask = df_raw[col].isnull() & df_clean[col].notnull()
            if mask.any():
                print(f"\nCột: {col}")
                sample = pd.concat([df_raw.loc[mask, col].head(3), df_clean.loc[mask, col].head(3)], axis=1)
                sample.columns = ['Before (NaN)', 'After (Filled)']
                print(sample)

# Audit theo row-group để không giữ đồng thời hai DataFrame 2.7 triệu dòng trong RAM.
import pyarrow.compute as pc
import pyarrow.parquet as pq

raw_file = pq.ParquetFile("../../data/mlmart_base/v5_preprocessing.parquet")
clean_file = pq.ParquetFile("../../data/mlmart_base/v5_final_cleaned.parquet")
common_cols = [c for c in raw_file.schema_arrow.names if c in clean_file.schema_arrow.names]
metric_cols = [c for c in [
    'shortwave_radiation', 'direct_normal_irradiance', 'diffuse_solar_radiation',
    'temperature_c', 'cloud_cover_total', 'cloud_cover_low', 'cloud_cover_mid',
    'cloud_cover_high', 'wind_speed', 'precipitation_mm', 'sunshine_duration'
] if c in common_cols]

def streaming_stats(parquet_file, columns):
    nulls = {c: 0 for c in columns}
    sums = {c: 0.0 for c in metric_cols}
    counts = {c: 0 for c in metric_cols}
    for batch in parquet_file.iter_batches(batch_size=100_000, columns=columns):
        for name, array in zip(batch.schema.names, batch.columns):
            nulls[name] += array.null_count
            if name in metric_cols:
                value = pc.sum(array).as_py()
                sums[name] += float(value or 0.0)
                counts[name] += len(array) - array.null_count
    means = {c: sums[c] / counts[c] if counts[c] else np.nan for c in metric_cols}
    return nulls, means

raw_nulls, raw_means = streaming_stats(raw_file, common_cols)
clean_nulls, clean_means = streaming_stats(clean_file, common_cols)
summary = pd.DataFrame({'Null_Before': raw_nulls, 'Null_After': clean_nulls})
summary['Filled_Count'] = summary['Null_Before'] - summary['Null_After']
print('--- TỔNG KẾT SỐ LƯỢNG GIÁ TRỊ ĐÃ FILL ---')
print(summary[summary['Filled_Count'] > 0].to_string())
stats_compare = pd.DataFrame({'Mean_Before': raw_means, 'Mean_After': clean_means})
stats_compare['Diff_%'] = ((stats_compare['Mean_After'] - stats_compare['Mean_Before']) / stats_compare['Mean_Before'] * 100).abs()
print('\n--- SO SÁNH MEAN (CHỈ CÁC CỘT METRIC) ---')
print(stats_compare.to_string())

--- TỔNG KẾT SỐ LƯỢNG GIÁ TRỊ ĐÃ FILL ---
                          Null_Before  Null_After  Filled_Count
panel                         1132078           0       1132078
inverter                      1132078           0       1132078
optimizers                    1259007           0       1259007
site_metric                   1132078           0       1132078
weather_type_id                   218           0           218
weather_is_day                    218           0           218
shortwave_radiation               218           0           218
direct_normal_irradiance          218           0           218
diffuse_solar_radiation           218           0           218
temperature_c                     218           0           218
cloud_cover_total                 218           0           218
cloud_cover_low                   218           0           218
cloud_cover_mid                   218           0           218
cloud_cover_high                  218           0           21